In [0]:
from pyspark.sql import functions as F

# Base S3 paths
S3_RAW_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = "s3://zubair-s3-demo/raw_dataset/aml/delta_tables"
CHECKPOINT_PATH = f"{S3_RAW_PATH}/checkpoints"

In [0]:

# ============================================================
# 3. Transactions - Incremental Auto Loader
# ============================================================
#
# Only NEW transaction files are processed after the first run.
#
# The Auto Loader checkpoint:
#   - remembers which files were already processed
#   - prevents previously processed files from being ingested again
#
# The Delta table remains the SAME:
#   aml_engine.aml_poc.bronze_transactions
#
# New transactions are APPENDED to it.
# ============================================================

raw_tx_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        f"{CHECKPOINT_PATH}/schema_tx"
    )
    .load(f"{S3_RAW_PATH}/transactions/")
)


# Add ingestion metadata
bronze_tx_df = (
    raw_tx_stream
    .withColumn(
        "_ingested_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
)


# Incremental append into the EXISTING Bronze table
query = (
    bronze_tx_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{CHECKPOINT_PATH}/bronze_tx"
    )
    .option(
        "path",
        f"{S3_DELTA_PATH}/bronze_transactions"
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        "aml_engine.aml_poc.bronze_transactions"
    )
)


query.awaitTermination()

